# 🏥 XRayGuard – Train on NIH ChestX-ray14

This notebook trains a **DenseNet121** classifier on 3000-5000 chest X-ray images from the [NIH ChestX-ray14](https://www.kaggle.com/datasets/nih-chest-xrays/data) dataset.

**Steps:**
1. Download dataset
2. Prepare data manifest
3. Train classifier
4. Calibrate & evaluate
5. Download trained weights

In [ ]:
# Install dependencies
!pip install -q kagglehub scikit-learn tqdm opencv-python-headless

In [ ]:
# Upload xrayguard project files (model.py, prepare_data.py, train.py, predict.py)
# Or clone from your repository
import os
WORK_DIR = '/content/xrayguard'
os.makedirs(WORK_DIR, exist_ok=True)

# If you uploaded the files to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/xrayguard/*.py /content/xrayguard/

# Or upload directly:
from google.colab import files
print('Upload model.py, prepare_data.py, train.py, predict.py:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'{WORK_DIR}/{name}', 'wb') as f:
        f.write(data)
    print(f'  Saved {name}')

In [ ]:
# Download NIH ChestX-ray14 dataset via kagglehub
import kagglehub
data_path = kagglehub.dataset_download('nih-chest-xrays/data')
print(f'Dataset downloaded to: {data_path}')

# List contents
!ls -la {data_path}/ | head -20

In [ ]:
# Step 1: Prepare data manifest (5000 images)
os.chdir(WORK_DIR)
!python prepare_data.py \
    --data_dir {data_path} \
    --max_images 5000 \
    --output /content/manifest.csv

In [ ]:
# Verify manifest
import pandas as pd
df = pd.read_csv('/content/manifest.csv')
print(f'Total images: {len(df)}')
print(f'Patients: {df.patient_id.nunique()}')
print(f'\nSplit distribution:')
print(df.split.value_counts())
print(f'\nLabel distribution:')
from model import LABELS
for l in LABELS:
    if l in df.columns:
        print(f'  {l}: {df[l].sum()}')

In [ ]:
# Step 2: Train the model (GPU recommended)
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!python train.py \
    --manifest /content/manifest.csv \
    --work_dir /content/weights \
    --epochs 15 \
    --bs 16 \
    --patience 4 \
    --weighted \
    --workers 2

In [ ]:
# Step 3: Review results
import json
r = json.load(open('/content/weights/results.json'))
print(f"Mean AUROC: {r['detection']['mean_AUROC']:.4f}")
print(f"95% CI: {r['detection']['CI95']}")
print(f"\nPer-finding AUROC:")
for k, v in r['detection']['per_finding'].items():
    print(f"  {k}: {v:.4f}")
print(f"\nCalibration ECE: {r['calibration']['ECE_before']:.4f} → {r['calibration']['ECE_after']:.4f}")
print(f"Temperature: {r['calibration']['temperature']:.4f}")

In [ ]:
# Step 4: Download trained weights
from google.colab import files
files.download('/content/weights/best_model.pth')
files.download('/content/weights/config.json')
files.download('/content/weights/results.json')
print('\n✓ Download complete! Place these files in xrayguard/weights/ on your local machine.')

In [ ]:
# (Optional) Test a prediction
os.chdir(WORK_DIR)
test_img = df[df.split == 'test'].img_path.iloc[0]
from predict import XRayPredictor
pred = XRayPredictor(work_dir='/content/weights')
result = pred.predict(test_img)
print(f"\nFindings: {result['findings']}")
print(f"Impression: {result['impression']}")
print(f"\nPresent: {result['findings_present']}")
print(f"Possible: {result['findings_possible']}")